# 02 — Silver: Customer & Identity

**Owner:** M2 • **Phase:** 4 (Thu 14 May lunch – Sun 17 May)

## Scope
- **Source tables (3):** `cl_customers` (2,501) • `cl_loyalty_tiers` (4) • `cl_loyalty_transactions` (2,860)
- **Transforms owned:** **T4** — Customer Identity Resolution (4-pass)
- **Silver tables you produce:** `silver_customer_master`, `silver_loyalty_tiers`, `silver_loyalty_transactions`
- **Anomaly territory:** A11 (placeholder ID 9999), B5 (cross-channel match)

## Hint script (no counts revealed)

Implement T4 in this strict order:
1. **Exact `loyalty_id`** match → `identity_confidence = 1.00`
2. **Exact email** (lowercase + trim) → `0.95`
3. **Exact phone** (E.164 normalised) → `0.90`
4. **Fuzzy name + address Jaccard** → `0.70 – 0.89` (use `rapidfuzz`)

Anonymous bucket: guest checkouts with no contact info → single shared SK + confidence 0.0.

**Hint A11:** something is weird about `customer_id = 9999`. Profile its order count vs the median. Cross-check: is there a real loyalty record with the same id? If the loyalty record has 0 actual transactions and the same id is used by N anonymous EC orders... what does that suggest?

**Hint B5:** when a customer has the same phone + address but different emails across channels, what's your defensible threshold? Document your choice in the report Section 7.

## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

## Cell 2 — Imports & config

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

## Cell 3 — `silver_loyalty_tiers` (small static lookup, easiest to start)

**TODO:**
- Read `cl_loyalty_tiers` from Bronze
- Generate SK on `tier_id`
- Add 4 mandatory anomaly columns (default CLEAN/CONFIRMED)
- Write `silver_loyalty_tiers`

In [ ]:
# TODO M2: see Cell 3 markdown for spec
tiers = read_bronze('cl_loyalty_tiers')
# tiers = tiers.withColumn('loyalty_tier_key', surrogate_key(F.col('tier_id')))
# tiers = add_anomaly_columns(tiers)
# write_silver(tiers, 'silver_loyalty_tiers')

## Cell 4 — `silver_loyalty_transactions`

**TODO:**
- Read `cl_loyalty_transactions` from Bronze
- Parse the date column (check format hint in `docs/date_formats.md`)
- Generate SK on `(customer_id, transaction_id)` (composite key)
- Add anomaly columns; flag any DATE_PARSE_FAIL
- Optionally: flag NEGATIVE_AMOUNT on point columns (a debit IS valid here, but anomalously large negatives might indicate import errors)
- Write `silver_loyalty_transactions`

In [ ]:
# TODO M2: see Cell 4 markdown for spec
loy_txn = read_bronze('cl_loyalty_transactions')
# ... your transforms ...

## Cell 5 — T4: 4-pass identity resolution

This is the centrepiece. Build `silver_customer_master` by unioning identity records from `cl_customers` (loyalty members) + EC guest customers + POS customers + NL user accounts, then resolving to canonical identities.

### Pass 1: exact loyalty_id match (confidence 1.00)
Customers in `cl_customers` have `loyalty_id`. EC orders sometimes carry the same `loyalty_id` if customer logged in. Join on this — exact matches are deterministic.

### Pass 2: exact email (confidence 0.95)
After lowercase + trim. Catch typos in your normalisation.

### Pass 3: exact phone in E.164 (confidence 0.90)
Strip non-digits, prepend country code. Indian numbers default to `+91`.

### Pass 4: fuzzy name + address Jaccard (confidence 0.70–0.89)
Use `rapidfuzz`. Tokenize names + address → Jaccard on token sets. Threshold 0.70.

### Anomaly handling
- Confidence 0.70–0.89 → `data_quality_status='FLAGGED_AMBIGUOUS'`, code `IDENTITY_AMBIGUOUS`
- Confidence < 0.70 → route to `anonymous_customer_key` (single shared SK), code `FUZZY_MATCH_LOW_CONF`
- **Customer ID 9999 collision**: detect specifically — code `PLACEHOLDER_ID_COLLISION`, status `FLAGGED_ANOMALY`

**TODO:** implement passes; produce `silver_customer_master` with one row per resolved customer carrying:
- `customer_master_key` (SHA-256 of best identifier)
- `identity_confidence`
- `match_pass_used` ∈ {1, 2, 3, 4, 0_anonymous}
- All identity attributes (email, phone, name, address) merged from highest-confidence source
- 4 mandatory anomaly columns

In [ ]:
# TODO M2: implement 4-pass identity resolution
# Recommended structure:
#
# 1. Build a unified "identity_candidate" DF unioning (cl_customers, ec_orders, pos_transactions, nl_user_accounts)
#    with a normalised schema: source_system, source_id, email_lower, phone_e164, name, address
#
# 2. Pass 1: window over loyalty_id, take min(source_id) as group leader
# 3. Pass 2: among unmatched, window over email_lower
# 4. Pass 3: among unmatched, window over phone_e164
# 5. Pass 4: blocking key (first 3 chars of name) + rapidfuzz on (name, address)
#
# 6. Anonymous bucket: anything still unmatched + no contact info → anonymous_key()
#
# 7. Detect placeholder-vs-real collision on customer_id:
#    Profile customer_id values in ec_orders by frequency. The most-used
#    placeholder (if any) will stand out. Then check cl_customers for a
#    real loyalty record sharing the same id. If both exist, that's the
#    collision — flag affected rows with PLACEHOLDER_ID_COLLISION.
#
customer_master = None  # placeholder
# customer_master = add_anomaly_columns(customer_master)
# customer_master = flag(customer_master,
#     condition=(F.col('identity_confidence').between(0.70, 0.89)),
#     reason_code='IDENTITY_AMBIGUOUS', status='FLAGGED_AMBIGUOUS', certainty='INFERRED')
# customer_master = flag(customer_master,
#     condition=(F.col('match_pass_used') == 0),
#     reason_code='FUZZY_MATCH_LOW_CONF', status='EXCLUDED_WITH_REASON', certainty='UNRELIABLE')
# customer_master = flag(customer_master,
#     condition=(F.col('detected_collision') == True),
#     reason_code='PLACEHOLDER_ID_COLLISION', status='FLAGGED_ANOMALY', certainty='UNRELIABLE')
# write_silver(customer_master, 'silver_customer_master')

## Cell 6 — Acceptance check (run before requesting PR review)

Self-validate before lead reviews:
1. Every row in `silver_customer_master` has all 4 mandatory columns populated (no nulls)
2. `identity_confidence` ranges across all 5 buckets (1.00, 0.95, 0.90, 0.70-0.89, 0.0)
3. Anonymous bucket has exactly ONE distinct `customer_master_key` value
4. Customer 9999 collision is flagged with `PLACEHOLDER_ID_COLLISION` (you can verify via Bronze: `SELECT COUNT(*) FROM ec_orders WHERE customer_id = 9999`; that count of EC rows should appear in your Silver as flagged)

In [ ]:
# Verification queries (uncomment after building)
# cm = spark.read.format('snowflake').options(**sf_silver).option('dbtable', 'silver_customer_master').load()
# 
# print('Identity confidence distribution:')
# cm.groupBy('identity_confidence').count().orderBy('identity_confidence').show()
# 
# print('Anomaly distribution:')
# cm.groupBy('data_quality_status', 'anomaly_reason_code').count().show(truncate=False)
# 
# print('Anonymous bucket SK count (should be 1):')
# cm.filter(F.col('match_pass_used') == 0).select('customer_master_key').distinct().count()

## Phase 5–6: your Gold dim

After Silver is accepted, build `dim_customer` (SCD2 + identity bridge) in `notebooks/03_gold_dimensions.ipynb` — see your section there.